In [1]:
#!/usr/bin/env python3
"""
CASE STUDY ANALYSIS - DeepNote Compatible
===================
Deep dive into best and worst predictions for each model.
"""

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend
from matplotlib import pyplot as plt
from matplotlib.figure import Figure
import seaborn as sns

# Set style
sns.set_style("whitegrid")

print("="*80)
print("CASE STUDY ANALYSIS: Best & Worst Predictions")
print("="*80)

# ================================================================
# 1. Load Data
# ================================================================

print("\n1. Loading data...")

df = pd.read_csv("predictions_all_stages_long.csv")

# Add continent
continent_mapping = {
    'Afghanistan': 'Asia', 'Albania': 'Europe', 'Algeria': 'Africa', 'Argentina': 'South America',
    'Armenia': 'Asia', 'Australia': 'Oceania', 'Austria': 'Europe', 'Bangladesh': 'Asia',
    'Belgium': 'Europe', 'Benin': 'Africa', 'Bolivia': 'South America', 'Bosnia Herzegovina': 'Europe',
    'Botswana': 'Africa', 'Brazil': 'South America', 'Bulgaria': 'Europe', 'Burkina Faso': 'Africa',
    'Cambodia': 'Asia', 'Cameroon': 'Africa', 'Canada': 'North America', 'Chad': 'Africa',
    'Chile': 'South America', 'China': 'Asia', 'Colombia': 'South America', 'Congo Brazzaville': 'Africa',
    'Costa Rica': 'North America', 'Croatia': 'Europe', 'Cyprus': 'Europe', 'Czech Republic': 'Europe',
    'Denmark': 'Europe', 'Dominican Republic': 'North America', 'Ecuador': 'South America', 'Egypt': 'Africa',
    'El Salvador': 'North America', 'Estonia': 'Europe', 'Ethiopia': 'Africa', 'Finland': 'Europe',
    'France': 'Europe', 'Gabon': 'Africa', 'Georgia': 'Asia', 'Germany': 'Europe',
    'Ghana': 'Africa', 'Greece': 'Europe', 'Guatemala': 'North America', 'Guinea': 'Africa',
    'Haiti': 'North America', 'Honduras': 'North America', 'Hong Kong': 'Asia', 'Hungary': 'Europe',
    'Iceland': 'Europe', 'India': 'Asia', 'Indonesia': 'Asia', 'Iran': 'Asia',
    'Iraq': 'Asia', 'Ireland': 'Europe', 'Israel': 'Asia', 'Italy': 'Europe',
    'Ivory Coast': 'Africa', 'Jamaica': 'North America', 'Japan': 'Asia', 'Jordan': 'Asia',
    'Kazakhstan': 'Asia', 'Kenya': 'Africa', 'Kosovo': 'Europe', 'Kyrgyzstan': 'Asia',
    'Laos': 'Asia', 'Latvia': 'Europe', 'Lebanon': 'Asia', 'Liberia': 'Africa', 'Libya': 'Africa',
    'Lithuania': 'Europe', 'Luxembourg': 'Europe', 'Macedonia': 'Europe', 'Madagascar': 'Africa',
    'Malawi': 'Africa', 'Malaysia': 'Asia', 'Mali': 'Africa', 'Malta': 'Europe',
    'Mauritania': 'Africa', 'Mauritius': 'Africa', 'Mexico': 'North America', 'Moldova': 'Europe',
    'Mongolia': 'Asia', 'Montenegro': 'Europe', 'Morocco': 'Africa', 'Mozambique': 'Africa',
    'Myanmar': 'Asia', 'Namibia': 'Africa', 'Nepal': 'Asia', 'Netherlands': 'Europe',
    'New Zealand': 'Oceania', 'Nicaragua': 'North America', 'Niger': 'Africa', 'Nigeria': 'Africa',
    'North Macedonia': 'Europe', 'Norway': 'Europe', 'Pakistan': 'Asia', 'Palestinian Territories': 'Asia',
    'Panama': 'North America', 'Paraguay': 'South America', 'Peru': 'South America', 'Philippines': 'Asia',
    'Poland': 'Europe', 'Portugal': 'Europe', 'Romania': 'Europe', 'Russia': 'Europe', 'Rwanda': 'Africa',
    'Saudi Arabia': 'Asia', 'Senegal': 'Africa', 'Serbia': 'Europe', 'Sierra Leone': 'Africa',
    'Singapore': 'Asia', 'Slovakia': 'Europe', 'Slovenia': 'Europe', 'South Africa': 'Africa',
    'South Korea': 'Asia', 'Spain': 'Europe', 'Sri Lanka': 'Asia', 'Sweden': 'Europe',
    'Switzerland': 'Europe', 'Taiwan': 'Asia', 'Tajikistan': 'Asia', 'Tanzania': 'Africa',
    'Thailand': 'Asia', 'Togo': 'Africa', 'Tunisia': 'Africa', 'Turkey': 'Asia',
    'Turkmenistan': 'Asia', 'Uganda': 'Africa', 'Ukraine': 'Europe', 'United Arab Emirates': 'Asia',
    'United Kingdom': 'Europe', 'United States': 'North America', 'Uruguay': 'South America',
    'Uzbekistan': 'Asia', 'Venezuela': 'South America', 'Vietnam': 'Asia', 'Yemen': 'Asia',
    'Zambia': 'Africa', 'Zimbabwe': 'Africa'
}
df['continent'] = df['countrynew'].map(continent_mapping)

# Load ground truth
gt_df = pd.read_csv("data_final.csv")
df = df.merge(gt_df, on='countrynew', how='left')
df['ground_truth_pi'] = df['mean_other_willingness'] * 100

# Focus on Stage 8
stage8_df = df[df['stage'] == 8].copy()
stage8_df = stage8_df.dropna(subset=['ground_truth_pi'])

print(f"✓ Loaded {len(stage8_df)} countries")

# Calculate errors
models = {
    'Llama': 'pred_llama',
    'GPT': 'pred_gpt',
    'Claude': 'pred_claude',
    'Gemini': 'pred_gemini'
}

for model_name, col in models.items():
    if col in stage8_df.columns:
        stage8_df[f'{col}_error'] = stage8_df[col] - stage8_df['ground_truth_pi']
        stage8_df[f'{col}_abs_error'] = np.abs(stage8_df[f'{col}_error'])

# ================================================================
# 2. Identify Best & Worst Cases
# ================================================================

print("\n" + "="*80)
print("2. IDENTIFYING BEST & WORST PREDICTIONS")
print("="*80)

case_studies = {}

for model_name, col in models.items():
    error_col = f'{col}_abs_error'
    if error_col not in stage8_df.columns:
        continue
    
    best = stage8_df.nsmallest(10, error_col)[['countrynew', 'continent', col, 
                                                'ground_truth_pi', error_col] + 
                                               [c for c in ['gdp_capita_2021', 'mean_religion', 
                                                           'mean_edu', 'hdi_2021'] 
                                                if c in stage8_df.columns]]
    
    worst = stage8_df.nlargest(10, error_col)[['countrynew', 'continent', col, 
                                                'ground_truth_pi', error_col] +
                                               [c for c in ['gdp_capita_2021', 'mean_religion', 
                                                           'mean_edu', 'hdi_2021'] 
                                                if c in stage8_df.columns]]
    
    case_studies[model_name] = {'best': best, 'worst': worst}
    
    print(f"\n{model_name}:")
    print(f"   Best prediction: {best.iloc[0]['countrynew']} (error: {best.iloc[0][error_col]:.2f}pp)")
    print(f"   Worst prediction: {worst.iloc[0]['countrynew']} (error: {worst.iloc[0][error_col]:.2f}pp)")

# ================================================================
# 3. Pattern Analysis
# ================================================================

print("\n" + "="*80)
print("3. PATTERN ANALYSIS")
print("="*80)

def analyze_group_characteristics(df, group_name):
    print(f"\n{group_name}:")
    if len(df) == 0:
        print("   No data")
        return
    print(f"   Continents: {dict(df['continent'].value_counts())}")
    numeric_cols = ['gdp_capita_2021', 'mean_religion', 'mean_edu', 'hdi_2021']
    available_cols = [c for c in numeric_cols if c in df.columns]
    if available_cols:
        print(f"   Characteristics:")
        for col in available_cols:
            if df[col].notna().sum() > 0:
                print(f"      {col}: {df[col].mean():.2f}")

for model_name in models.keys():
    print(f"\n{'='*60}")
    print(f"{model_name}")
    print('='*60)
    if model_name in case_studies:
        analyze_group_characteristics(case_studies[model_name]['best'], f"Best 10 predictions")
        analyze_group_characteristics(case_studies[model_name]['worst'], f"Worst 10 predictions")

# ================================================================
# 4-6. Other Analysis (omitted for brevity - add back if needed)
# ================================================================

# Calculate standard deviation
model_cols = [col for col in ['pred_llama', 'pred_gpt', 'pred_claude', 'pred_gemini'] 
             if col in stage8_df.columns]
stage8_df['prediction_std'] = stage8_df[model_cols].std(axis=1)

# Export
with open('case_study_report.txt', 'w') as f:
    f.write("="*80 + "\n")
    f.write("CASE STUDY ANALYSIS: Best & Worst Predictions\n")
    f.write("="*80 + "\n\n")

for model_name in models.keys():
    if model_name in case_studies:
        case_studies[model_name]['best'].to_csv(f'case_study_{model_name.lower()}_best.csv', index=False)
        case_studies[model_name]['worst'].to_csv(f'case_study_{model_name.lower()}_worst.csv', index=False)

print("\n✓ Saved case study CSVs")

# ================================================================
# 7. Visualizations
# ================================================================

print("\n" + "="*80)
print("7. CREATING VISUALIZATIONS")
print("="*80)

# Figure 1: Best vs Worst characteristics
fig = plt.figure()
fig.set_size_inches(14, 10)
fig.set_dpi(300)
matplotlib.rcParams.update({})

# Create subplots
ax1 = fig.add_subplot(2, 2, 1)
ax2 = fig.add_subplot(2, 2, 2)
ax3 = fig.add_subplot(2, 2, 3)
ax4 = fig.add_subplot(2, 2, 4)
axes = [ax1, ax2, ax3, ax4]

features_to_plot = ['gdp_capita_2021', 'mean_religion', 'mean_edu', 'hdi_2021']
feature_names = ['GDP per Capita', 'Religious Importance', 'Education (years)', 'HDI']

for idx, (feat, name) in enumerate(zip(features_to_plot, feature_names)):
    if feat not in stage8_df.columns:
        continue
    
    ax = axes[idx]
    
    for model_name in models.keys():
        if model_name not in case_studies:
            continue
        
        best_vals = case_studies[model_name]['best'][feat].dropna()
        worst_vals = case_studies[model_name]['worst'][feat].dropna()
        
        if len(best_vals) > 0 and len(worst_vals) > 0:
            ax.scatter([model_name] * len(best_vals), best_vals, 
                      color='green', alpha=0.6, s=100, marker='o', label='Best' if idx == 0 else '')
            ax.scatter([model_name] * len(worst_vals), worst_vals, 
                      color='red', alpha=0.6, s=100, marker='x', label='Worst' if idx == 0 else '')
    
    ax.set_ylabel(name, fontsize=11, fontweight='bold')
    ax.set_title(f'{name}: Best vs Worst Predictions', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    if idx == 0:
        ax.legend(fontsize=10)

fig.suptitle('Characteristics of Best vs Worst Predictions by Model', 
            fontsize=15, fontweight='bold', y=0.995)
fig.tight_layout()
fig.savefig('case_study_characteristics.png', dpi=300, bbox_inches='tight')
fig.savefig('case_study_characteristics.pdf', bbox_inches='tight')
print("✓ Saved case_study_characteristics.png")
print("✓ Saved case_study_characteristics.pdf")
plt.close()

# Figure 2: Model agreement
fig2 = plt.figure()
fig2.set_size_inches(12, 8)
fig2.set_dpi(300)
matplotlib.rcParams.update({})
ax = fig2.add_subplot(1, 1, 1)

disagreement_sorted = stage8_df.nlargest(20, 'prediction_std')

y_pos = np.arange(len(disagreement_sorted))
ax.barh(y_pos, disagreement_sorted['prediction_std'], color='coral', alpha=0.7)
ax.set_yticks(y_pos)
ax.set_yticklabels(disagreement_sorted['countrynew'], fontsize=9)
ax.set_xlabel('Standard Deviation of Predictions (pp)', fontsize=12, fontweight='bold')
ax.set_title('Countries with Highest Model Disagreement\n(Large std = models predict very differently)', 
            fontsize=14, fontweight='bold', pad=20)
ax.grid(True, alpha=0.3, axis='x')

fig2.tight_layout()
fig2.savefig('model_disagreement.png', dpi=300, bbox_inches='tight')
fig2.savefig('model_disagreement.pdf', bbox_inches='tight')
print("✓ Saved model_disagreement.png")
print("✓ Saved model_disagreement.pdf")
plt.close()

print("\n" + "="*80)
print("CASE STUDY ANALYSIS COMPLETE!")
print("="*80)

print("\nFiles created:")
print("   - case_study_report.txt")
print("   - case_study_[model]_best.csv (for each model)")
print("   - case_study_[model]_worst.csv (for each model)")
print("   - case_study_characteristics.png / .pdf")
print("   - model_disagreement.png / .pdf")

CASE STUDY ANALYSIS: Best & Worst Predictions

1. Loading data...
✓ Loaded 125 countries

2. IDENTIFYING BEST & WORST PREDICTIONS

Llama:
   Best prediction: Sweden (error: 0.10pp)
   Worst prediction: Gabon (error: 23.32pp)

GPT:
   Best prediction: Russia (error: 0.57pp)
   Worst prediction: Israel (error: 30.57pp)

Claude:
   Best prediction: Australia (error: 0.02pp)
   Worst prediction: Israel (error: 23.07pp)

Gemini:
   Best prediction: Honduras (error: 0.28pp)
   Worst prediction: Afghanistan (error: 35.98pp)

3. PATTERN ANALYSIS

Llama

Best 10 predictions:
   Continents: {'Europe': 4, 'Africa': 3, 'Asia': 2, 'Oceania': 1}
   Characteristics:
      gdp_capita_2021: 27291.65
      mean_religion: 0.62
      mean_edu: 0.15
      hdi_2021: 0.76

Worst 10 predictions:
   Continents: {'Europe': 4, 'Africa': 2, 'North America': 2, 'Asia': 1, 'South America': 1}
   Characteristics:
      gdp_capita_2021: 24286.84
      mean_religion: 0.80
      mean_edu: 0.11
      hdi_2021: 0.74

GPT

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=feb9f195-de2a-416f-b8f1-09efca4e954f' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>